# SDT full-parameter DPO: training and evaluation

This notebook runs the complete 100-record demonstration pipeline: repository setup, GPU verification, preference-pair preparation, tests, **full-parameter DPO**, locked baseline and DPO evaluation, paired comparison, and optional Google Drive backup.

The 100 records are sufficient to verify that the pipeline works, but not to establish a reliable alignment improvement. Run the cells in order in a fresh GPU runtime.

## 1. Confirm that Colab assigned a GPU
Select **Runtime → Change runtime type → GPU** before running this cell.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "No CUDA GPU is available. Change the Colab runtime to GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone or update the repository
For a private repository, create a Colab secret named `GITHUB_TOKEN`, paste a GitHub token with read access, and enable notebook access to that secret. A public repository needs no token.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Rana-Ezzeddine/SDT.git"
REPO_DIR = Path("/content/SDT")

github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

git = ["git"]
if github_token:
    git += ["-c", f"http.extraHeader=AUTHORIZATION: bearer {github_token}"]

if not REPO_DIR.exists():
    try:
        subprocess.run(git + ["clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Clone failed. If the repository is private, add a GITHUB_TOKEN Colab secret."
        ) from exc
elif (REPO_DIR / ".git").exists():
    subprocess.run(git + ["-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Use a fresh runtime.")

os.chdir(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Working directory:", Path.cwd())
subprocess.run(["git", "status", "--short", "--branch"], check=True)

## 2b. Apply the validated TRL compatibility fix
This makes the notebook safe even if the GitHub clone still contains the earlier trainer version. It removes the redundant uncached evaluation call that caused the completed run to stop, and replaces the deprecated warmup ratio with two warmup steps. The cell does nothing when the repository is already fixed.

In [ ]:
from pathlib import Path

train_path = Path("src/sdt_dpo/train.py")
train_code = train_path.read_text()
old_evaluate = (
    '    validation_metrics = trainer.evaluate(\n'
    '        eval_dataset=validation_dpo,\n'
    '        metric_key_prefix="validation",\n'
    '    )'
)
new_evaluate = '    validation_metrics = trainer.evaluate(metric_key_prefix="validation")'
train_code = train_code.replace(old_evaluate, new_evaluate)
train_code = train_code.replace(
    'warmup_ratio=float(config.get("warmup_ratio", 0.1)),',
    'warmup_steps=int(config.get("warmup_steps", 2)),',
)
train_path.write_text(train_code)

config_path = Path("configs/full.yaml")
config_text = config_path.read_text().replace("warmup_ratio: 0.10", "warmup_steps: 2")
config_path.write_text(config_text)

assert 'trainer.evaluate(metric_key_prefix="validation")' in train_path.read_text()
assert "warmup_steps: 2" in config_path.read_text()
assert "warmup_ratio" not in config_path.read_text()
print("TRL compatibility fix confirmed.")

## 3. Install the project
The public Qwen checkpoint will be downloaded automatically later; no Hugging Face token is required.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 4. Inspect the full-DPO configuration
The learning rate is intentionally much lower than a LoRA learning rate. `precompute_ref_log_probs` reduces memory use by caching the frozen reference model's scores before training.

In [ ]:
from pathlib import Path

print(Path("configs/full.yaml").read_text())

## 5. Prepare the chosen/rejected pairs
Failed judge blocks are excluded rather than changed to score zero. The split is assigned at the prompt level before creating multiple response pairs.

In [ ]:
import subprocess

subprocess.run([
    "sdt-build-pairs",
    "--input", "data/raw/sdt_100_llama.json",
    "--output", "data/processed/dpo_pairs.jsonl",
    "--report", "data/processed/pair_report.json",
    "--min-common-judges", "2",
    "--min-margin", "0.10",
    "--min-confidence", "0.60",
    "--train-share", "0.70",
    "--validation-share", "0.15",
    "--seed", "42",
], check=True)

In [ ]:
import json
from pathlib import Path

pair_report = json.loads(Path("data/processed/pair_report.json").read_text())
summary_fields = [
    "records", "possible_pairs", "retained_pairs", "retained_prompts",
    "retained_by_split", "retained_by_confidence",
    "retained_by_comparison_type", "exclusion_reasons",
]
for field in summary_fields:
    print(f"{field}: {pair_report[field]}")

assert pair_report["retained_by_split"] == {"train": 139, "validation": 27, "test": 22}
print("Sample preparation counts match the audited pipeline.")

## 6. Run repository tests
These tests cover failed-judge handling, pair direction, prompt leakage, full-parameter configuration, and comparison statistics.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## 7. Train the full-parameter DPO model
This downloads `Qwen/Qwen2.5-0.5B-Instruct`, verifies that all 494M parameters are trainable, precomputes reference log-probabilities, trains for one epoch, validates, and saves a complete model. Expect several minutes on a Colab GPU.

Do not use the test results to change training settings after this point. If CUDA runs out of memory, start a fresh runtime, change `max_length: 1024` to `512`, and rerun from the beginning.

In [ ]:
import subprocess

subprocess.run(["sdt-train-dpo", "--config", "configs/full.yaml"], check=True)

## 8. Locate and verify the trained model
A successful fixed script saves the final model in `outputs/dpo-full`. The checkpoint fallback also lets this notebook recover a model from a run that completed training but stopped during final reporting.

In [ ]:
from pathlib import Path

final_model = Path("outputs/dpo-full")
has_final_weights = (final_model / "config.json").exists() and any(final_model.glob("*.safetensors"))

if has_final_weights:
    DPO_MODEL = str(final_model)
else:
    checkpoints = sorted(
        final_model.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    assert checkpoints, "Training produced neither a final model nor a checkpoint."
    DPO_MODEL = str(checkpoints[-1])

print("DPO model for evaluation:", DPO_MODEL)
print("Model files:")
for path in sorted(Path(DPO_MODEL).glob("*")):
    if path.is_file():
        print(" -", path.name)

## 9. Evaluate the unchanged baseline on the locked test pairs
The evaluator calculates length-normalized conditional response log-probability. A pair is correct when the model assigns a higher score to the judge-preferred response.

In [ ]:
import subprocess

subprocess.run([
    "sdt-evaluate-pairs",
    "--pairs", "data/processed/dpo_pairs.jsonl",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--split", "test",
    "--max-length", "1024",
    "--output", "outputs/base-test.json",
    "--details", "outputs/base-test-pairs.jsonl",
], check=True)

## 10. Evaluate the full-DPO model on exactly the same test pairs

In [ ]:
import subprocess

subprocess.run([
    "sdt-evaluate-pairs",
    "--pairs", "data/processed/dpo_pairs.jsonl",
    "--model", DPO_MODEL,
    "--split", "test",
    "--max-length", "1024",
    "--output", "outputs/dpo-test.json",
    "--details", "outputs/dpo-test-pairs.jsonl",
], check=True)

## 11. Verify that evaluation is paired
This guard prevents an invalid comparison if either evaluator skipped every pair or the files came from different data splits.

In [ ]:
import json
from pathlib import Path

def load_details(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

base_rows = load_details("outputs/base-test-pairs.jsonl")
dpo_rows = load_details("outputs/dpo-test-pairs.jsonl")
base_ids = {str(row["pair_id"]) for row in base_rows}
dpo_ids = {str(row["pair_id"]) for row in dpo_rows}
shared_ids = base_ids & dpo_ids

print("Baseline evaluated pairs:", len(base_ids))
print("DPO evaluated pairs:", len(dpo_ids))
print("Shared pair IDs:", len(shared_ids))

if not shared_ids:
    for summary_path in ["outputs/base-test.json", "outputs/dpo-test.json"]:
        report = json.loads(Path(summary_path).read_text())
        print(summary_path, "overall=", report.get("overall"), "skipped=", report.get("skipped"))
    raise RuntimeError("No shared test pairs. Inspect the printed skipped reasons before comparing.")

assert base_ids == dpo_ids, "The evaluators did not score exactly the same pair IDs."
print("Pairing check passed.")

## 12. Produce the paired baseline-versus-DPO report

In [ ]:
import subprocess

subprocess.run([
    "sdt-compare-evaluations",
    "--baseline-details", "outputs/base-test-pairs.jsonl",
    "--dpo-details", "outputs/dpo-test-pairs.jsonl",
    "--output", "outputs/base-vs-dpo.json",
], check=True)

## 13. Display the main results
Positive DPO-minus-baseline deltas favor DPO. `prompt_macro_accuracy` gives every prompt equal weight. The prompt-cluster bootstrap interval is the primary uncertainty interval because several pairs can come from one prompt. With only 10 test prompts, expect wide uncertainty.

In [ ]:
import json
import pandas as pd
from pathlib import Path

comparison = json.loads(Path("outputs/base-vs-dpo.json").read_text())
display(pd.DataFrame([
    {
        "model": "Baseline",
        "pair_accuracy": comparison["baseline_preference_accuracy"],
        "prompt_macro_accuracy": comparison["baseline_prompt_macro_accuracy"],
        "mean_model_margin": comparison["mean_model_margin_baseline"],
    },
    {
        "model": "Full DPO",
        "pair_accuracy": comparison["dpo_preference_accuracy"],
        "prompt_macro_accuracy": comparison["dpo_prompt_macro_accuracy"],
        "mean_model_margin": comparison["mean_model_margin_dpo"],
    },
]))

print("Pair-accuracy delta:", comparison["accuracy_delta_dpo_minus_baseline"])
print("Prompt-macro delta:", comparison["prompt_macro_accuracy_delta"])
print("Prompt-cluster bootstrap 95% interval:", comparison["prompt_cluster_bootstrap_95"])
print("Mean-margin delta:", comparison["mean_model_margin_delta"])
print("Discordant pairs:", comparison["discordant_pairs"])

## 14. Save the complete outputs to Google Drive
Colab runtime storage is temporary. Run this cell after evaluation to preserve the full checkpoint and reports. Each run gets a timestamped directory, so previous results are not overwritten.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
from google.colab import drive

drive.mount("/content/drive")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destination = Path("/content/drive/MyDrive/SDT_full_DPO_results") / timestamp
shutil.copytree("outputs", destination)
print("Saved outputs to:", destination)